# Whisper Karşılaştırma — Tüm Kombinasyonlar

Tek çalıştırmada bir model için tüm config varyasyonlarını test eder:
- Voice isolation: açık / kapalı
- Timestamp: stable-ts / none

Sonuç: 4 farklı SRT dosyası (karşılaştırma için)

## Bu notebook hangi model?
**large-v3** — Genel amaçlı Whisper

kotoba-whisper için `colab_whisper_compare_kotoba.ipynb` kullan.

## Kullanım
A: Kurulum → B: Dosya yükle → C: Otomatik tüm kombinasyonlar → D: SRT indir

---
# A) Kurulum

In [ ]:
import os
import subprocess
import time

import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı!')
print(f'\u2705 GPU: {torch.cuda.get_device_name(0)}')

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('\u2705 HF_TOKEN')
except Exception:
    print('\u26a0\ufe0f HF_TOKEN yok')

!pip install -q faster-whisper stable-ts demucs
!apt-get -qq install ffmpeg > /dev/null 2>&1
print('\u2705 Kurulum tamam')

---
# B) Dosya Yükle

In [ ]:
from google.colab import files
from pathlib import Path

print('Dosya seç:')
uploaded = files.upload()
INPUT_FILE = list(uploaded.keys())[0]
STEM = Path(INPUT_FILE).stem
print(f'\u2705 {INPUT_FILE}')

# Ses çıkar
AUDIO_FILE = 'audio.wav'
if INPUT_FILE.lower().endswith(('.mp4', '.mkv', '.avi', '.webm', '.mov')):
    subprocess.run(
        ['ffmpeg', '-y', '-i', INPUT_FILE, '-vn', '-acodec', 'pcm_s16le', '-ar', '16000', '-ac', '1', AUDIO_FILE],
        capture_output=True, check=True,
    )
else:
    AUDIO_FILE = INPUT_FILE
print(f'\u2705 Ses: {AUDIO_FILE}')

# Demucs için 44.1kHz stereo versiyon
DEMUCS_INPUT = 'audio_44k.wav'
subprocess.run(
    ['ffmpeg', '-y', '-i', AUDIO_FILE, '-ar', '44100', '-ac', '2', DEMUCS_INPUT],
    capture_output=True, check=True,
)

# Vocal izolasyon (bir kere yapılır, tüm testlerde kullanılır)
print('\U0001f3b5 Vocal izolasyon (Demucs)...')
t0 = time.time()
result = subprocess.run(
    ['python', '-m', 'demucs', '--two-stems', 'vocals', '-n', 'htdemucs_ft', DEMUCS_INPUT],
    capture_output=True, text=True,
)
demucs_stem = os.path.splitext(os.path.basename(DEMUCS_INPUT))[0]
VOCAL_FILE = f'separated/htdemucs_ft/{demucs_stem}/vocals.wav'
if result.returncode == 0 and os.path.exists(VOCAL_FILE):
    print(f'\u2705 Vocal izole ({time.time()-t0:.1f}s)')
else:
    VOCAL_FILE = None
    print(f'\u274c Demucs başarısız:\n{result.stderr[-300:]}')

---
# C) Tüm Kombinasyonları Çalıştır

In [ ]:
import gc
import time

WHISPER_MODEL = 'large-v3'

# Test matrisi
CONFIGS = []
for vocal_iso in [False, True]:
    for ts_method in ['none', 'stable-ts']:
        if vocal_iso and VOCAL_FILE is None:
            continue  # Demucs başarısız olduysa atla
        CONFIGS.append({'vocal_iso': vocal_iso, 'ts_method': ts_method})

print(f'\U0001f9ea Model: {WHISPER_MODEL}')
print(f'   {len(CONFIGS)} kombinasyon çalıştırılacak:\n')
for i, c in enumerate(CONFIGS, 1):
    print(f'   {i}. vocal_iso={c["vocal_iso"]}, timestamp={c["ts_method"]}')
print()


def ts(sec):
    h, m, s, ms = int(sec//3600), int(sec%3600//60), int(sec%60), int(sec%1*1000)
    return f'{h:02d}:{m:02d}:{s:02d},{ms:03d}'


all_results = []

for idx, cfg in enumerate(CONFIGS, 1):
    vocal_iso = cfg['vocal_iso']
    ts_method = cfg['ts_method']

    input_audio = VOCAL_FILE if vocal_iso else AUDIO_FILE
    tag = f"{'vocal' if vocal_iso else 'raw'}_{ts_method}"
    srt_name = f'{STEM}_largev3_{tag}.srt'

    print(f'\n{"="*60}')
    print(f'[{idx}/{len(CONFIGS)}] {tag}')
    print(f'  input: {input_audio}, method: {ts_method}')
    print(f'{"="*60}')

    t0 = time.time()
    segments = []

    if ts_method == 'stable-ts':
        import stable_whisper
        model = stable_whisper.load_faster_whisper(WHISPER_MODEL)
        result = model.transcribe_stable(
            input_audio, task='transcribe', language='ja', beam_size=5, vad=True,
        )
        for seg in result.segments:
            segments.append({'start': seg.start, 'end': seg.end, 'text': seg.text.strip()})
    else:
        from faster_whisper import WhisperModel
        model = WhisperModel(WHISPER_MODEL, device='cuda', compute_type='float16')
        segs, _ = model.transcribe(
            input_audio, task='transcribe', language='ja', beam_size=5,
            vad_filter=True, vad_parameters=dict(min_silence_duration_ms=300, speech_pad_ms=200),
        )
        for seg in segs:
            segments.append({'start': seg.start, 'end': seg.end, 'text': seg.text.strip()})

    elapsed = time.time() - t0
    print(f'  \u2705 {len(segments)} segment, {elapsed:.1f}s')

    # SRT yaz
    with open(srt_name, 'w', encoding='utf-8') as f:
        for i, s in enumerate(segments, 1):
            f.write(f'{i}\n{ts(s["start"])} --> {ts(s["end"])}\n{s["text"]}\n\n')

    all_results.append({'tag': tag, 'srt': srt_name, 'segments': len(segments), 'time': elapsed})

    # GPU temizle
    del model
    torch.cuda.empty_cache()
    gc.collect()

# Özet
print(f'\n\n{"="*60}')
print(f'ÖZET — {WHISPER_MODEL}')
print(f'{"="*60}')
print(f'{"Tag":<25} {"Segment":>8} {"Süre":>8}')
print(f'{"-"*25} {"-"*8} {"-"*8}')
for r in all_results:
    print(f'{r["tag"]:<25} {r["segments"]:>8} {r["time"]:>7.1f}s')

---
# D) Tüm SRT İndir

In [ ]:
from google.colab import files

print('SRT dosyaları:')
for r in all_results:
    print(f'  \u2705 {r["srt"]}')
    files.download(r['srt'])
print(f'\n\U0001f4e6 {len(all_results)} dosya indirildi')